In [1]:
import os

In [ ]:
os.environ["OPENAI_API_KEY"]="open ai api key"
os.environ["LANGCHAIN_API_KEY"]="langchain api key"

In [3]:
from openai import OpenAI

from openevals.llm import create_llm_as_judge
from openevals.prompts import CORRECTNESS_PROMPT

correctness_evaluator = create_llm_as_judge(
    prompt=CORRECTNESS_PROMPT,
    model="gpt-4o-mini",
    judge=OpenAI(),
    continuous=True
)

f:\Projects\Langgraph\lang\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# inputs = "How much has the price of doodads changed in the past year?"
# outputs = "Doodads have increased in price by 10% in the past year."
# reference_outputs = "The price of doodads has decreased by 50% in the past year."

# eval_result = correctness_evaluator(
#   inputs=inputs,
#   outputs=outputs,
#   reference_outputs=reference_outputs
# )

# print(eval_result)

In [ ]:
from openevals.llm import create_llm_as_judge
from openevals.prompts import HALLUCINATION_PROMPT


Hallucination_evaluator = create_llm_as_judge(
    prompt=HALLUCINATION_PROMPT,
    model="gpt-4o-mini",
    judge=OpenAI(),
    continuous=True
)

# eval_result = Hallucination_evaluator(inputs=inputs, outputs=outputs, context=context,reference_outputs="abc")

In [6]:
from openevals.llm import create_llm_as_judge
from openevals.prompts import CONCISENESS_PROMPT

inputs = "How is the weather in San Francisco?"
outputs = "Thanks for asking! The current weather in San Francisco is sunny and 90 degrees."

conciseness_evaluator= create_llm_as_judge(
    prompt=CONCISENESS_PROMPT,
    model="gpt-4o-mini",
    judge=OpenAI(),
    continuous=True
)

# eval_result = conciseness_evaluator(inputs=inputs, outputs=outputs)

# print(eval_result)

In [7]:
from openevals.string.embedding_similarity import create_embedding_similarity_evaluator

similarity_evaluator = create_embedding_similarity_evaluator()

# result = evaluator(
#     outputs="The weather is nice!",
#     reference_outputs="The weather is very nice!",
# )

# print(result)

In [ ]:
import pandas as pd
df=pd.read_excel("our_RAG7.xlsx")

In [39]:
df

,Question,Ground truth,context,answer
0,What does Article 150 of the Constitution of I...,Article 150 of the Constitution of India provi...,1.1.1 Article 150 of the Constitution of Ind...,Article 150 of the Constitution of India provi...
1,Where does the payment process in PFMS start?,The payment process in PFMS starts at Program ...,2.2 PROCESSING OF CLAIMS IN PAOs THROUGH PFMS ...,The payment process in PFMS starts at Program ...
2,What must be done when discrepancies are notic...,"Any discrepancies noticed, in the PFMS-CAM rep...",PFMS under ‘CAM Reports’ viz. “PAY -02: Sancti...,Any discrepancies noticed in the PFMS-CAM repo...
3,What should be done when cheques remain un-enc...,The particulars of the cheques outstanding/rem...,accounts against it. \n \n2.19.5 The part...,The particulars of the cheques outstanding/rem...
4,Who prepares a consolidated monthly account fo...,Office of CGA prepares a consolidated monthly ...,No.S.11019/App.4/78/TA/4652 dated 28.7.1979)...,The Office of CGA prepares a consolidated mont...
5,Who maintains the G.P.F. accounts of All India...,The PAO in ‘Accountant General’s (AGs) Office ...,Telecommunication Department and other State a...,The PAO in 'Accountant General's (AGs) office ...
6,What is the mandatory contribution rate under ...,"Under Tier-I, the Government servants have to ...",scheme viz. Defined Pension Contribution Schem...,The mandatory contribution rate under Tier-I o...
7,Who is responsible for maintaining the detaile...,The Principal Accounts Offices/PAO Secretariat...,already made which were due to the Central Gov...,The Principal Accounts Offices /PAO Secretaria...
8,What does the Finance Accounts of the Central ...,The Finance Accounts of the Central Government...,284 CHAPTER 1 2 \n \nFINANCE ACCOUNTS \n \...,The Finance Accounts of the Central Government...
9,What does Statement No. 6 in Part II of the Fi...,"In the first half of the statement, the total ...","summarized statements in respect of Revenue, C...",Statement No. 6 in Part II of the Finance Acco...


In [40]:
correctness=0
conciseness=0
hallucination=0
similarity=0
for row in df.values.tolist():
  correctness_result = correctness_evaluator(
  inputs=row[0],
  outputs=row[3],
  reference_outputs=row[1])

  conciseness_result = conciseness_evaluator(inputs=row[0], outputs=row[3])

  hallucination_result= Hallucination_evaluator(inputs=row[0], outputs=row[3], context=row[2],reference_outputs=row[1])

  similarity_result=result = similarity_evaluator(
    outputs=row[3],
    reference_outputs=row[1])
  
  correctness+=correctness_result["score"]
  conciseness+=conciseness_result["score"]
  hallucination+=hallucination_result["score"]
  similarity+=similarity_result["score"]


  
    



In [41]:
print("correctness=",correctness*100/20," ","conciseness=", conciseness*100/20," ","hallucination=", hallucination*100/20," ","similarity=", similarity*100/20)

correctness= 76.50000000000001   conciseness= 89.0   hallucination= 88.50000000000001   similarity= 76.25


In [8]:
print(HALLUCINATION_PROMPT)

You are an expert data labeler evaluating model outputs for hallucinations. Your task is to assign a score based on the following rubric:

<Rubric>
  A response without hallucinations:
  - Contains only verifiable facts that are directly supported by the input context
  - Makes no unsupported claims or assumptions
  - Does not add speculative or imagined details
  - Maintains perfect accuracy in dates, numbers, and specific details
  - Appropriately indicates uncertainty when information is incomplete
</Rubric>

<Instructions>
  - Read the input context thoroughly
  - Identify all claims made in the output
  - Cross-reference each claim with the input context
  - Note any unsupported or contradictory information
  - Consider the severity and quantity of hallucinations
</Instructions>

<Reminder>
  Focus solely on factual accuracy and support from the input context. Do not consider style, grammar, or presentation in scoring. A shorter, factual response should score higher than a longer 

In [4]:
print(CORRECTNESS_PROMPT)

You are an expert data labeler evaluating model outputs for correctness. Your task is to assign a score based on the following rubric:

<Rubric>
  A correct answer:
  - Provides accurate and complete information
  - Contains no factual errors
  - Addresses all parts of the question
  - Is logically consistent
  - Uses precise and accurate terminology

  When scoring, you should penalize:
  - Factual errors or inaccuracies
  - Incomplete or partial answers
  - Misleading or ambiguous statements
  - Incorrect terminology
  - Logical inconsistencies
  - Missing key information
</Rubric>

<Instructions>
  - Carefully read the input and output
  - Check for factual accuracy and completeness
  - Focus on correctness of information rather than style or verbosity
</Instructions>

<Reminder>
  The goal is to evaluate factual correctness and completeness of the response.
</Reminder>

<input>
{inputs}
</input>

<output>
{outputs}
</output>

Use the reference outputs below to help you evaluate the

In [7]:
print(CONCISENESS_PROMPT)

You are an expert data labeler evaluating model outputs for conciseness. Your task is to assign a score based on the following rubric:

<Rubric>
  A perfectly concise answer:
  - Contains only the exact information requested.
  - Uses the minimum number of words necessary to convey the complete answer.
  - Omits pleasantries, hedging language, and unnecessary context.
  - Excludes meta-commentary about the answer or the model's capabilities.
  - Avoids redundant information or restatements.
  - Does not include explanations unless explicitly requested.

  When scoring, you should deduct points for:
  - Introductory phrases like "I believe," "I think," or "The answer is."
  - Hedging language like "probably," "likely," or "as far as I know."
  - Unnecessary context or background information.
  - Explanations when not requested.
  - Follow-up questions or offers for more information.
  - Redundant information or restatements.
  - Polite phrases like "hope this helps" or "let me know if y

In [9]:
HALLUCINATION_PROMPT

'You are an expert data labeler evaluating model outputs for hallucinations. Your task is to assign a score based on the following rubric:\n\n<Rubric>\n  A response without hallucinations:\n  - Contains only verifiable facts that are directly supported by the input context\n  - Makes no unsupported claims or assumptions\n  - Does not add speculative or imagined details\n  - Maintains perfect accuracy in dates, numbers, and specific details\n  - Appropriately indicates uncertainty when information is incomplete\n</Rubric>\n\n<Instructions>\n  - Read the input context thoroughly\n  - Identify all claims made in the output\n  - Cross-reference each claim with the input context\n  - Note any unsupported or contradictory information\n  - Consider the severity and quantity of hallucinations\n</Instructions>\n\n<Reminder>\n  Focus solely on factual accuracy and support from the input context. Do not consider style, grammar, or presentation in scoring. A shorter, factual response should score 

In [10]:
CONCISENESS_PROMPT

'You are an expert data labeler evaluating model outputs for conciseness. Your task is to assign a score based on the following rubric:\n\n<Rubric>\n  A perfectly concise answer:\n  - Contains only the exact information requested.\n  - Uses the minimum number of words necessary to convey the complete answer.\n  - Omits pleasantries, hedging language, and unnecessary context.\n  - Excludes meta-commentary about the answer or the model\'s capabilities.\n  - Avoids redundant information or restatements.\n  - Does not include explanations unless explicitly requested.\n\n  When scoring, you should deduct points for:\n  - Introductory phrases like "I believe," "I think," or "The answer is."\n  - Hedging language like "probably," "likely," or "as far as I know."\n  - Unnecessary context or background information.\n  - Explanations when not requested.\n  - Follow-up questions or offers for more information.\n  - Redundant information or restatements.\n  - Polite phrases like "hope this helps" 

In [11]:
CORRECTNESS_PROMPT

'You are an expert data labeler evaluating model outputs for correctness. Your task is to assign a score based on the following rubric:\n\n<Rubric>\n  A correct answer:\n  - Provides accurate and complete information\n  - Contains no factual errors\n  - Addresses all parts of the question\n  - Is logically consistent\n  - Uses precise and accurate terminology\n\n  When scoring, you should penalize:\n  - Factual errors or inaccuracies\n  - Incomplete or partial answers\n  - Misleading or ambiguous statements\n  - Incorrect terminology\n  - Logical inconsistencies\n  - Missing key information\n</Rubric>\n\n<Instructions>\n  - Carefully read the input and output\n  - Check for factual accuracy and completeness\n  - Focus on correctness of information rather than style or verbosity\n</Instructions>\n\n<Reminder>\n  The goal is to evaluate factual correctness and completeness of the response.\n</Reminder>\n\n<input>\n{inputs}\n</input>\n\n<output>\n{outputs}\n</output>\n\nUse the reference 